# 06 · Study 1: model and predicted absorption of candidates

Model used to predict fractional iron absorption for the candidates of evaluation study 1
(proof-of-concept evaluation, Methods). The model is trained on the oral supplement measurements of
the pooled dataset with the predictors available at study 1 screening (13 predictors, including hepcidin), using the same
procedure as the main model (80:20 split, grid search with five-fold cross-validation in the training
set). Predictions are generated with the saved model `../models/model_4studyA.json`; candidates with a
predicted fractional absorption above 20% are classified as predicted high absorbers.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, train_test_split

## 1. Training data: oral supplement measurements of the pooled dataset

In [2]:
KEEP = ["Age (y)", "Fe dose", "Gender", "BV", "Fe.circ", "MC.freq", "MC.duration", "Hb", "Hep",
"Ferritin", "sTfR", "C-reactive protein", "Alpha glycoprotein", "Meal type", "Fe compound"]

raw = pd.read_excel("../data/pooled_dataset.xls")
d = raw[KEEP + ["Fe absorption (%)"]].copy()
d["Meal type"] = d["Meal type"].fillna("Supplement")
for col in d.columns:
    if d[col].dtype != float:
        d[col] = d[col].astype("category").cat.codes
d = d[d["Meal type"] == 36].drop(columns=["Gender", "Meal type"]).reset_index(drop=True)

y = np.log10(d["Fe absorption (%)"] + 1)
X = d[[c for c in d.columns if c != "Fe absorption (%)"]]
print("Supplement measurements:", len(d), "| predictors:", X.shape[1])
print(list(X.columns))

Supplement measurements: 1244 | predictors: 13
['Age (y)', 'Fe dose', 'BV', 'Fe.circ', 'MC.freq', 'MC.duration', 'Hb', 'Hep', 'Ferritin', 'sTfR', 'C-reactive protein', 'Alpha glycoprotein', 'Fe compound']


## 2. Model training

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
param_grid = {"n_estimators": [300, 400], "learning_rate": [0.01, 0.05], "max_depth": [8, 9], "subsample": [0.8, 0.9], "colsample_bytree": [0.5, 0.6, 0.7]}
grid_search = GridSearchCV(xgb.XGBRegressor(objective="reg:squarederror"), param_grid, cv=5,
                           scoring="neg_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Selected hyperparameters:", grid_search.best_params_)

Selected hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 400, 'subsample': 0.8}


## 3. Predicted absorption of the study 1 candidates (saved model)

In [4]:
model = xgb.XGBRegressor()
model.load_model("../models/model_4studyA.json")
assert model.get_booster().feature_names == list(X.columns)

cand = pd.read_excel("../data/study1_candidates.xlsx")
cand["Meal type"] = 36   # oral supplement
cand["Fe compound"] = 9  # ferrous sulphate
cand = cand[[c for c in KEEP if c not in ("Gender", "Meal type")]]

predicted = (10 ** model.predict(cand).astype(float)) - 1   # back-transformed to fractional absorption (%)
group = np.where(predicted > 20, "Predicted high absorber", "Predicted low absorber")
summary = (pd.DataFrame({"group": group, "predicted": predicted})
           .groupby("group")["predicted"].agg(n="size", median="median", min="min", max="max").round(1))
print("Candidates:", len(cand))
print(summary.to_string())

Candidates: 45
                          n  median   min   max
group                                          
Predicted high absorber  27    23.0  20.2  33.0
Predicted low absorber   18    18.4   8.4  19.8
